## Imports

In [1]:
import pandas as pd
import numpy as np

## Load data

In [4]:
df = pd.read_csv("../data/raw/synthetic_transactions.csv")
df.head()

,transaction_id,account_id,user_age,account_age_days,transaction_amount,transaction_hour,ip_risk_score,num_prev_transactions_24h,avg_transaction_amount_7d,failed_login_attempts_24h,email_domain,device_type,payment_method,country,is_foreign_transaction,shipping_billing_mismatch,kyc_completed,has_chargeback_history,is_synthetic_account,is_fraud
0,TXN_502215,ACC_102215,30,604,1338.75,13,0.194,2,1171.43,0,icloud.com,mobile,card,Sweden,no,no,yes,no,no,0
1,TXN_502582,ACC_102582,37,879,1354.23,23,0.144,3,1112.89,0,gmail.com,mobile,card,Norway,no,no,yes,no,no,0
2,TXN_501662,ACC_101662,26,30,305.43,18,0.271,4,236.11,0,gmail.com,mobile,card,Norway,no,no,yes,no,no,0
3,TXN_503027,ACC_103027,40,529,160.37,7,0.418,1,666.58,0,outlook.com,desktop,card,Germany,no,no,yes,no,no,0
4,TXN_504343,ACC_104343,41,779,678.02,8,0.256,1,1011.00,0,icloud.com,mobile,apple_pay,Romania,no,no,yes,no,no,0


## Shape + columns
What does the training data contain?

In [5]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (8000, 20)

Columns:
['transaction_id', 'account_id', 'user_age', 'account_age_days', 'transaction_amount', 'transaction_hour', 'ip_risk_score', 'num_prev_transactions_24h', 'avg_transaction_amount_7d', 'failed_login_attempts_24h', 'email_domain', 'device_type', 'payment_method', 'country', 'is_foreign_transaction', 'shipping_billing_mismatch', 'kyc_completed', 'has_chargeback_history', 'is_synthetic_account', 'is_fraud']


## info()

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 20 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   transaction_id             8000 non-null   object 
 1   account_id                 8000 non-null   object 
 2   user_age                   8000 non-null   int64  
 3   account_age_days           8000 non-null   int64  
 4   transaction_amount         8000 non-null   float64
 5   transaction_hour           8000 non-null   int64  
 6   ip_risk_score              8000 non-null   float64
 7   num_prev_transactions_24h  8000 non-null   int64  
 8   avg_transaction_amount_7d  8000 non-null   float64
 9   failed_login_attempts_24h  8000 non-null   int64  
 10  email_domain               8000 non-null   object 
 11  device_type                8000 non-null   object 
 12  payment_method             8000 non-null   object 
 13  country                    8000 non-null   objec

## Target balance
Checks what percentage of training data is fraud

In [7]:
df["is_fraud"].value_counts(normalize=True)

is_fraud
0    0.9
1    0.1
Name: proportion, dtype: float64

## Fraud vs. Legit
Compare the average values ​​between the scammers and the genuine accounts

In [8]:
df.groupby("is_fraud")[
    [
        "transaction_amount",
        "account_age_days",
        "ip_risk_score",
        "failed_login_attempts_24h",
        "num_prev_transactions_24h",
    ]
].mean()

,transaction_amount,account_age_days,ip_risk_score,failed_login_attempts_24h,num_prev_transactions_24h
is_fraud,,,,,
0,857.770136,426.628333,0.182688,0.401111,1.98250
1,4405.301312,230.632500,0.732130,2.547500,5.24375


## Synthetic accounts
How does the model analyze the content?

In [9]:
pd.crosstab(df["is_synthetic_account"], df["is_fraud"], normalize="index")

is_fraud,0,1
is_synthetic_account,,
no,0.962696,0.037304
yes,0.000000,1.000000


## Suspicious countries
Of the fraudsters found in the training data, where do they come from?

In [10]:
pd.crosstab(df["country"], df["is_fraud"]).sort_values(by=1, ascending=False)

is_fraud,0,1
country,,
Poland,203,105
Nigeria,58,98
Sweden,2588,85
Germany,868,79
Romania,145,74
Norway,1056,64
Russia,71,62
Finland,730,61
Denmark,705,53


## Summary

In [11]:
print("Fraud rate:", round(df["is_fraud"].mean()*100,2), "%")
print("Average amount:", round(df["transaction_amount"].mean(),2))
print("Average IP risk:", round(df["ip_risk_score"].mean(),2))

Fraud rate: 10.0 %
Average amount: 1212.52
Average IP risk: 0.24
